In [ ]:
from pathlib import Path
import sys

module_path = Path.cwd().parent.parent.absolute()

if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from fedotllm.main import FedotAI
from fedotllm.handlers import JupyterOutput

dataset_path = Path.cwd() / "competition"
dataset_path.mkdir(parents=True, exist_ok=True)

/Users/aleksejlapin/Work/STABLE-FedotLLM/.venv/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


2025-07-23 17:47:58,424 - HTTP Request: GET https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json "HTTP/1.1 200 OK"


In [ ]:
import shutil

output_path = Path.cwd() / 'output'
if output_path.exists():
    shutil.rmtree(output_path)
output_path.mkdir(parents=True, exist_ok=True)

In [3]:
description = \
"""
Your Goal:
Each instance contains 8 features: airline carrier code, flight number, origin airport, destination airport, day of week (1-7), time of departure (in minutes), flight duration
(in minutes), and the binary target variable indicating delay status (0 for on-time, 1 for delayed). Good luck!
Evaluation
The evaluation metric for this binary classification task is ROC AUC.
Submission File
Target: "Delay" column.
"""

In [4]:
fedot_ai = FedotAI(
        task_path=dataset_path,
        workspace=output_path,
        handlers=JupyterOutput().subscribe
    )

async for _ in fedot_ai.ask(message=description):
    continue

================== HumanMessage ==================


Your Goal:
Each instance contains 8 features: airline carrier code, flight number, origin airport, destination airport, day of week (1-7), time of departure (in minutes), flight duration
(in minutes), and the binary target variable indicating delay status (0 for on-time, 1 for delayed). Good luck!
Evaluation
The evaluation metric for this binary classification task is ROC AUC.
Submission File
Target: "Delay" column.


================== HumanMessage ==================

# Understanding Flight Delay Predictions: A Machine Learning Overview

## **Overview**
- **Problem Description**: Flight delays can disrupt travel plans, leading to frustration for passengers and financial losses for airlines. Predicting whether a flight will be delayed based on various features (like the airline, time of departure, and others) is crucial for managing these disruptions.
- **Goal**: Our model aims to accurately forecast flight delays, helping airlines and passengers make more informed decisions.

## **Data Preprocessing**
Before diving into the modeling, we need to clean and prepare our data. Here's how we approached it:

- **Imputation**: 
  - Missing values were filled in using averages for numerical columns. For example, if the 'Flight Duration' had missing entries, these were replaced with the average flight duration.
  - Categorical variables with missing values were filled with the most common occurrence. If 'Airline' had missing values, we inserted the most frequently occuring airline.
  
- **Normalization**: 
  - While not specifically applied in our use case, if it were, normalization would ensure all features are comparable by rescaling them to a 0-1 range, which helps the model learn better.

- **Feature Selection**: 
  - We focused on essential features such as 'Airline', 'AirportFrom', 'AirportTo', 'DayOfWeek', 'Time', and 'Length'. These features help the model understand the conditions under which delays may occur.

## **Pipeline Summary**
The modeling process consisted of several key steps:

1. **Logistic Regression**: This model serves as a baseline to better understand our data.
2. **CatBoost**: 
   - Parameters Used: 
     | Model   | Parameters                                     | Explanation                                                |
     |---------|------------------------------------------------|------------------------------------------------------------|
     | CatBoost| `num_trees: 3000`, `learning_rate: 0.03`, `max_depth: 5`, `l2_leaf_reg: 0.01` | Chosen for its capability to handle categorical data effectively. |
3. **XGBoost**: Known for its speed and performance on structured data.
4. **LightGBM (LGBM)**: Extremely efficient for large datasets due to its gradient-based learning approach.
5. **Scaling**: No specific scaling applied; the models are robust to varying feature scales.

## **Code Highlights**
The code defines a structured approach to loading data, preprocessing, training, evaluating, and making predictions. Key functions include:

### Data Transformation
```python
def transform_data(dataset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    ...
    numeric_imputer = SimpleImputer(strategy="mean")
    categorical_imputer = SimpleImputer(strategy="most_frequent")
    ...
    return features.values, target
```
- This function separates features from the target variable and imputes missing values appropriately to ensure a clean dataset for training.

### Model Training
```python
def train_model(train_features: np.ndarray | pd.DataFrame, train_target: np.ndarray | pd.DataFrame | pd.Series):
    model = Fedot(problem=TaskTypesEnum.classification.value, ...)
    model.fit(features=input_data)
    ...
    return model
```
- The model fitting process is handled here, where the training features and target variable are provided. The `Fedot` framework automates the model training with optimal configurations.

### Evaluation
```python
def evaluate_model(model, test_features: np.ndarray | pd.DataFrame, test_target: np.ndarray | pd.DataFrame | pd.Series):
    ...
    y_pred = model.predict_proba(features=input_data)
    print("Model metrics: ", model.get_metrics())
    ...
    return model.get_metrics()
```
- This part evaluates model performance on the test dataset and returns key metrics.

### Submission File
```python
output.to_csv(SUBMISSION_PATH, index=False)
```
- Finally, predictions are saved to a CSV file for submission, structuring the results in an easily shareable format.

## **Metrics**
The performance metrics yielded a **ROC AUC of 0.715**. 
- **ROC AUC** is a vital measure of the model's ability to distinguish between classes. A score of 0.715 indicates that the model is fairly capable; it can correctly classify 71.5% of the instances, which is promising for flight delay prediction.

## **Takeaways**
- This model accurately predicts flight delays with a ROC AUC of 0.715, indicating it has significant potential in operational settings.
- By leveraging this model, airlines can enhance their efficiency and passenger satisfaction, even in the face of unpredictable delays, ultimately leading to better travel experiences. 

In summary, our machine-learning approach to predicting flight delays holds substantial promise. By understanding the underlying patterns, airlines can better manage flights and reduce passenger frustration, making air travel more reliable.